# ✂️ Colab 1 — Chunking
### Materia: Procesamiento de Lenguaje Natural — Nivel Terciario

---

## ¿Qué es el chunking y por qué importa?

Antes de indexar documentos en un vector store, hay que dividirlos en fragmentos más pequeños llamados **chunks**. Esta decisión parece trivial pero es una de las que más afecta la calidad del retrieval.

**¿Por qué no indexar el documento completo?**
- Los embeddings de textos muy largos "promedian" el significado y pierden precisión
- El retrieval devuelve documentos completos: si el dato relevante está en el párrafo 12, traés también los 11 anteriores al prompt
- Más tokens en el contexto = más costo y más ruido para el LLM

**¿Por qué no hacer chunks muy chicos?**
- Un chunk de 2 oraciones puede no tener suficiente contexto para ser útil
- La misma idea puede quedar partida en dos chunks y no recuperarse completa

## Las variables que vamos a explorar

| Variable | Opciones a comparar |
|---|---|
| **Tamaño** | 200 / 500 / 1000 caracteres |
| **Overlap** | 0% vs 20% de superposición |
| **Estrategia** | Fijo por chars / Por oración / Por párrafo |

> 🎯 **Objetivo del colab:** ver empíricamente cómo cada decisión afecta qué chunks se recuperan ante la misma pregunta.

In [1]:
!pip install chromadb sentence-transformers openai --quiet
print('✅ Dependencias instaladas')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 751.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the 

In [2]:
import os
from openai import OpenAI

try:
    from google.colab import userdata
    api_key = userdata.get('OPENAI_API_KEY')
except Exception:
    api_key = os.getenv('OPENAI_API_KEY', 'TU_API_KEY_ACA')

client = OpenAI(api_key=api_key)
print('✅ Cliente OpenAI inicializado')

✅ Cliente OpenAI inicializado


---
## 📄 El corpus: Reglamento Interno de TechStore Argentina

Usamos un documento largo con secciones variadas. Es importante que sea extenso para que el chunking tenga efecto real.

💡 **Al final del notebook hay una celda para reemplazar este corpus con el tuyo propio.**

In [3]:
CORPUS = """REGLAMENTO INTERNO DE OPERACIONES — TECHSTORE ARGENTINA S.A.
Versión 3.2 — Actualizado enero 2025

SECCIÓN 1: POLÍTICA DE DEVOLUCIONES Y CAMBIOS

1.1 Plazo de devolución
Los clientes tienen derecho a solicitar la devolución de productos dentro de los 30 días corridos contados desde la fecha de entrega. Para productos con defectos de fábrica, el plazo se extiende a 90 días. Las devoluciones iniciadas fuera de estos plazos no serán aceptadas bajo ninguna circunstancia, salvo que medie una orden judicial o resolución de organismo de defensa del consumidor.

1.2 Condiciones del producto para devolución
El producto debe presentarse en su embalaje original, sin señales de uso, con todos los accesorios incluidos y con el ticket o factura de compra. Los productos que hayan sido activados, configurados o que presenten signos evidentes de uso serán derivados al servicio técnico para evaluación antes de procesar cualquier devolución. Los productos de higiene personal, auriculares in-ear y colchones no admiten devolución por razones sanitarias.

1.3 Proceso de devolución
El cliente debe iniciar el proceso de devolución a través del portal web en la sección Mi Cuenta, o comunicándose con el centro de atención al cliente. Una vez aprobada la solicitud, el cliente recibirá un código de autorización (RMA) que deberá incluir en el paquete. Las devoluciones sin código RMA serán rechazadas automáticamente en el depósito. El reembolso se acredita en la misma forma de pago original dentro de los 10 días hábiles.

SECCIÓN 2: GARANTÍAS

2.1 Garantía legal
Todos los productos comercializados por TechStore Argentina tienen garantía legal mínima de 6 meses por defectos de fabricación, conforme a la Ley 24.240 de Defensa del Consumidor. Esta garantía es obligatoria y no puede ser reducida ni eliminada por ningún acuerdo contractual entre las partes.

2.2 Garantías extendidas por categoría
Los televisores de 40 pulgadas o más tienen garantía de 12 meses. Las notebooks, tablets y computadoras de escritorio tienen garantía de 12 meses. Los smartphones de gama alta (precio superior a $200.000) tienen garantía de 12 meses. Los electrodomésticos de línea blanca tienen garantía de 12 meses. Los accesorios, cables y periféricos tienen garantía de 3 meses.

2.3 Exclusiones de garantía
La garantía no cubre daños causados por mal uso, golpes, líquidos, modificaciones no autorizadas, o uso en condiciones distintas a las especificadas por el fabricante. Tampoco cubre el desgaste natural de la batería en dispositivos móviles cuando la capacidad se mantiene por encima del 80% de la original. Los daños por sobretensión eléctrica solo están cubiertos si el cliente puede demostrar que utilizaba un estabilizador de tensión certificado.

2.4 Proceso de garantía
Para hacer efectiva la garantía, el cliente debe presentar el producto en cualquier sucursal o enviarlo al centro de servicio técnico con flete prepago. El tiempo de reparación estimado es de 15 días hábiles. Si la reparación no puede completarse en ese plazo, TechStore Argentina ofrecerá un equipo de reemplazo de características similares o el reembolso del precio de compra a elección del cliente.

SECCIÓN 3: POLÍTICA DE ENVÍOS

3.1 Zonas de cobertura
TechStore Argentina realiza envíos a todo el territorio nacional. Las zonas clasificadas como remotas o de difícil acceso pueden tener plazos y costos adicionales. Se considera zona remota a las localidades que no cuenten con servicio de correo regular o que estén a más de 50 kilómetros del centro de distribución más cercano.

3.2 Plazos de entrega
Para CABA y GBA el plazo de entrega es de 24 a 48 horas hábiles. Para el interior del país el plazo es de 3 a 7 días hábiles dependiendo de la provincia de destino. Patagonia y NOA tienen un plazo de 7 a 12 días hábiles. Estos plazos son estimativos y pueden verse afectados por condiciones climáticas, feriados nacionales o situaciones de fuerza mayor.

3.3 Costos de envío
El envío es gratuito para compras iguales o superiores a $50.000 en todo el país. Para compras menores, el costo de envío se calcula en función del peso del paquete y la distancia al destino, y se informa al cliente antes de confirmar la compra. Los productos de más de 30 kg o con volumen superior a 1 metro cúbico tienen un cargo adicional de logística pesada.

3.4 Seguimiento y recepción
Una vez despachado el pedido, el cliente recibirá un número de seguimiento por email y SMS. El cliente debe estar presente en el domicilio indicado para recibir el paquete. Si no hay nadie al momento de la entrega, el transportista dejará un aviso y realizará un segundo intento al día hábil siguiente. Después de dos intentos fallidos, el paquete volverá al depósito y se cobrarán gastos de reenvío.

SECCIÓN 4: MEDIOS DE PAGO Y FINANCIACIÓN

4.1 Tarjetas aceptadas
Se aceptan todas las tarjetas de crédito de las redes Visa, Mastercard, American Express y Cabal. También se aceptan tarjetas de débito de las mismas redes. Las tarjetas prepagas solo se aceptan para compras de hasta $30.000.

4.2 Cuotas sin interés
El programa de cuotas sin interés varía según el banco emisor de la tarjeta y el período promocional vigente. Con tarjetas Visa y Mastercard se ofrecen 3, 6 y 12 cuotas sin interés en productos seleccionados. Los productos con descuento especial o en liquidación no participan del programa de cuotas sin interés. Las cuotas sin interés no son acumulables con otros descuentos o promociones.

4.3 Transferencia bancaria
Los pagos por transferencia bancaria reciben un descuento adicional del 10% sobre el precio de lista. La transferencia debe acreditarse dentro de las 48 horas de realizada la compra, caso contrario la orden será cancelada automáticamente. El descuento por transferencia no aplica para productos ya en promoción.

4.4 Billeteras digitales
Se aceptan pagos con Mercado Pago, Modo, Ualá y Naranja X. Los pagos con billeteras digitales tienen las mismas condiciones que las tarjetas de débito. Ocasionalmente pueden existir promociones específicas con alguna billetera digital que serán comunicadas en el sitio web.

SECCIÓN 5: ATENCIÓN AL CLIENTE Y RECLAMOS

5.1 Canales de contacto
El centro de atención al cliente opera de lunes a viernes de 9:00 a 18:00 horas y los sábados de 9:00 a 13:00 horas. Los canales disponibles son: WhatsApp al +54 11 4000-0000, email a soporte@techstore.com.ar, chat en vivo en el sitio web, y atención presencial en todas las sucursales durante el horario comercial.

5.2 Tiempos de respuesta
Los mensajes de WhatsApp y chat en vivo tienen un tiempo de respuesta máximo de 2 horas durante el horario de atención. Los emails se responden dentro de las 24 horas hábiles. Los reclamos formales se resuelven en un plazo máximo de 72 horas hábiles desde su ingreso al sistema. En casos complejos que requieran investigación interna, el plazo puede extenderse hasta 10 días hábiles, notificando al cliente.

5.3 Escalamiento de reclamos
Si el cliente no está satisfecho con la resolución de su reclamo, puede solicitar que sea escalado al área de Supervisión de Atención al Cliente. Si la insatisfacción persiste, el cliente tiene derecho a presentar su reclamo ante la Dirección de Defensa del Consumidor de su provincia o ante la plataforma nacional de resolución de conflictos. TechStore Argentina se compromete a responder todos los reclamos formales presentados ante organismos públicos dentro de los plazos legales establecidos.
"""

print(f"📄 Corpus cargado")
print(f"   Longitud total: {len(CORPUS)} caracteres")
print(f"   Palabras aproximadas: {len(CORPUS.split())}")

📄 Corpus cargado
   Longitud total: 7434 caracteres
   Palabras aproximadas: 1191


---
# 🔬 Experimento 1 — Tamaño de chunk

Vamos a indexar el mismo corpus tres veces con distintos tamaños y ver qué recupera cada uno ante la misma pregunta.

In [4]:
def chunk_fijo(texto, tamanio, overlap=0):
    """Divide el texto en chunks de tamaño fijo con overlap opcional."""
    chunks = []
    paso = tamanio - overlap
    inicio = 0
    while inicio < len(texto):
        fin = inicio + tamanio
        chunk = texto[inicio:fin].strip()
        if chunk:
            chunks.append(chunk)
        inicio += paso
    return chunks

for tamanio in [200, 500, 1000]:
    chunks = chunk_fijo(CORPUS, tamanio)
    print(f"📦 Tamaño {tamanio} chars → {len(chunks)} chunks | avg {sum(len(c) for c in chunks) // len(chunks)} chars")

📦 Tamaño 200 chars → 38 chunks | avg 195 chars
📦 Tamaño 500 chars → 15 chunks | avg 495 chars
📦 Tamaño 1000 chars → 8 chunks | avg 929 chars


In [5]:
# Veamos cómo quedan los chunks: ¿el texto se corta en medio de una oración?
print("🔍 Primeros 3 chunks con tamaño 200:")
print("=" * 60)
for i, chunk in enumerate(chunk_fijo(CORPUS, 200)[:3]):
    print(f"\n[Chunk {i+1}]")
    print(chunk)
    print("-" * 40)

print("\n⚠️  El chunking fijo corta en medio de ideas.")
print("   Esto puede partir información relevante entre dos chunks.")

🔍 Primeros 3 chunks con tamaño 200:

[Chunk 1]
REGLAMENTO INTERNO DE OPERACIONES — TECHSTORE ARGENTINA S.A.
Versión 3.2 — Actualizado enero 2025

SECCIÓN 1: POLÍTICA DE DEVOLUCIONES Y CAMBIOS

1.1 Plazo de devolución
Los clientes tienen derecho a
----------------------------------------

[Chunk 2]
solicitar la devolución de productos dentro de los 30 días corridos contados desde la fecha de entrega. Para productos con defectos de fábrica, el plazo se extiende a 90 días. Las devoluciones iniciad
----------------------------------------

[Chunk 3]
as fuera de estos plazos no serán aceptadas bajo ninguna circunstancia, salvo que medie una orden judicial o resolución de organismo de defensa del consumidor.

1.2 Condiciones del producto para devol
----------------------------------------

⚠️  El chunking fijo corta en medio de ideas.
   Esto puede partir información relevante entre dos chunks.


In [6]:
import chromadb
from chromadb.utils import embedding_functions

openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=api_key,
    model_name="text-embedding-3-small"
)

def crear_coleccion(nombre, chunks):
    chroma_client = chromadb.Client()
    try:
        chroma_client.delete_collection(nombre)
    except:
        pass
    col = chroma_client.create_collection(nombre, embedding_function=openai_ef)
    col.add(
        documents=chunks,
        ids=[f"{nombre}_chunk_{i}" for i in range(len(chunks))]
    )
    return col

def recuperar(coleccion, pregunta, top_k=2):
    resultados = coleccion.query(query_texts=[pregunta], n_results=top_k)
    return list(zip(resultados['documents'][0], resultados['distances'][0]))

print("⏳ Indexando tres tamaños distintos...")
col_200  = crear_coleccion("chunks_200",  chunk_fijo(CORPUS, 200))
col_500  = crear_coleccion("chunks_500",  chunk_fijo(CORPUS, 500))
col_1000 = crear_coleccion("chunks_1000", chunk_fijo(CORPUS, 1000))
print("✅ Las tres colecciones están listas")

⏳ Indexando tres tamaños distintos...
✅ Las tres colecciones están listas


In [7]:
def comparar_tamanios(pregunta, top_k=1):
    print(f"❓ Pregunta: {pregunta}")
    print("=" * 70)
    for nombre, col in [("200 chars", col_200), ("500 chars", col_500), ("1000 chars", col_1000)]:
        resultados = recuperar(col, pregunta, top_k)
        chunk, dist = resultados[0]
        print(f"\n📦 Chunks de {nombre} (distancia: {dist:.4f}):")
        print("-" * 40)
        print(chunk[:300] + ("..." if len(chunk) > 300 else ""))

comparar_tamanios("¿Cuántos días tengo para hacer una devolución?")
print()
comparar_tamanios("¿Qué pasa si nadie está en casa cuando llega el envío?")

❓ Pregunta: ¿Cuántos días tengo para hacer una devolución?

📦 Chunks de 200 chars (distancia: 0.6821):
----------------------------------------
solicitar la devolución de productos dentro de los 30 días corridos contados desde la fecha de entrega. Para productos con defectos de fábrica, el plazo se extiende a 90 días. Las devoluciones iniciad

📦 Chunks de 500 chars (distancia: 0.7814):
----------------------------------------
dicial o resolución de organismo de defensa del consumidor.

1.2 Condiciones del producto para devolución
El producto debe presentarse en su embalaje original, sin señales de uso, con todos los accesorios incluidos y con el ticket o factura de compra. Los productos que hayan sido activados, configur...

📦 Chunks de 1000 chars (distancia: 0.7635):
----------------------------------------
REGLAMENTO INTERNO DE OPERACIONES — TECHSTORE ARGENTINA S.A.
Versión 3.2 — Actualizado enero 2025

SECCIÓN 1: POLÍTICA DE DEVOLUCIONES Y CAMBIOS

1.1 Plazo de devolución
Los client

### 🔍 ¿Qué observás?

- El chunk de **200 chars** puede recuperar el dato exacto pero sin contexto suficiente
- El chunk de **500 chars** suele ser el punto de equilibrio para documentos de este tipo
- El chunk de **1000 chars** trae mucha información pero a veces mezcla temas distintos

> 💡 No existe un tamaño "correcto". Depende del tipo de documento y del tipo de preguntas esperadas.

---
# 🔬 Experimento 2 — El Overlap

El **overlap** es la superposición entre chunks consecutivos. Sirve para evitar que información quede partida entre dos chunks.

```
SIN OVERLAP:
[--chunk1--][--chunk2--][--chunk3--]

CON OVERLAP 20%:
[----chunk1----]
         [----chunk2----]
                  [----chunk3----]
```

In [8]:
overlap_20 = int(500 * 0.20)  # 100 chars

chunks_sin_overlap = chunk_fijo(CORPUS, 500, overlap=0)
chunks_con_overlap = chunk_fijo(CORPUS, 500, overlap=overlap_20)

print(f"Sin overlap:  {len(chunks_sin_overlap)} chunks")
print(f"Con overlap:  {len(chunks_con_overlap)} chunks")
print(f"Diferencia:   {len(chunks_con_overlap) - len(chunks_sin_overlap)} chunks extra")
print(f"\n💡 El overlap aumenta la cantidad de chunks pero mejora la cobertura en los bordes.")

Sin overlap:  15 chunks
Con overlap:  19 chunks
Diferencia:   4 chunks extra

💡 El overlap aumenta la cantidad de chunks pero mejora la cobertura en los bordes.


In [9]:
print("⏳ Indexando colecciones de overlap...")
col_sin_overlap = crear_coleccion("sin_overlap", chunks_sin_overlap)
col_con_overlap = crear_coleccion("con_overlap", chunks_con_overlap)
print("✅ Listo")

# Pregunta diseñada para estar en la zona de borde entre dos chunks
pregunta = "¿Qué pasa si la reparación en garantía tarda más de 15 días?"

print(f"\n❓ Pregunta: {pregunta}")
print("=" * 70)

for nombre, col in [("Sin overlap", col_sin_overlap), ("Con overlap 20%", col_con_overlap)]:
    resultados = recuperar(col, pregunta, top_k=1)
    chunk, dist = resultados[0]
    print(f"\n📦 {nombre} (distancia: {dist:.4f}):")
    print("-" * 40)
    print(chunk[:400] + ("..." if len(chunk) > 400 else ""))

⏳ Indexando colecciones de overlap...
✅ Listo

❓ Pregunta: ¿Qué pasa si la reparación en garantía tarda más de 15 días?

📦 Sin overlap (distancia: 0.9335):
----------------------------------------
tería en dispositivos móviles cuando la capacidad se mantiene por encima del 80% de la original. Los daños por sobretensión eléctrica solo están cubiertos si el cliente puede demostrar que utilizaba un estabilizador de tensión certificado.

2.4 Proceso de garantía
Para hacer efectiva la garantía, el cliente debe presentar el producto en cualquier sucursal o enviarlo al centro de servicio técnico c...

📦 Con overlap 20% (distancia: 0.8953):
----------------------------------------
requieran investigación interna, el plazo puede extenderse hasta 10 días hábiles, notificando al cliente.

5.3 Escalamiento de reclamos
Si el cliente no está satisfecho con la resolución de su reclamo, puede solicitar que sea escalado al área de Supervisión de Atención al Cliente. Si la insatisfacción persiste, el cl

---
# 🔬 Experimento 3 — Estrategias de chunking

El chunking fijo por caracteres es el más simple pero el más "tonto": no respeta la estructura del texto.

| Estrategia | Descripción | Cuándo usarla |
|---|---|---|
| **Fijo por chars** | Corta cada N caracteres | Textos sin estructura clara |
| **Por oración** | Agrupa oraciones | Textos narrativos |
| **Por párrafo** | Cada párrafo es un chunk | Documentos formales estructurados |
| **Recursivo** | Intenta párrafo → oración → char | Uso general, default de LangChain |

In [10]:
import re

def chunk_por_oracion(texto, oraciones_por_chunk=4, overlap_oraciones=1):
    oraciones = re.split(r'(?<=[.!?])\s+', texto.strip())
    oraciones = [o.strip() for o in oraciones if o.strip()]
    chunks = []
    paso = oraciones_por_chunk - overlap_oraciones
    for i in range(0, len(oraciones), paso):
        grupo = oraciones[i:i + oraciones_por_chunk]
        if grupo:
            chunks.append(" ".join(grupo))
    return chunks

def chunk_por_parrafo(texto, max_chars=600):
    parrafos = [p.strip() for p in texto.split('\n') if p.strip()]
    chunks = []
    buffer = ""
    for parrafo in parrafos:
        if len(buffer) + len(parrafo) < max_chars:
            buffer += (" " if buffer else "") + parrafo
        else:
            if buffer:
                chunks.append(buffer)
            buffer = parrafo
    if buffer:
        chunks.append(buffer)
    return chunks

chunks_fijo    = chunk_fijo(CORPUS, 500)
chunks_oracion = chunk_por_oracion(CORPUS)
chunks_parrafo = chunk_por_parrafo(CORPUS)

print("📊 Comparación de estrategias:")
print(f"  Fijo (500 chars):    {len(chunks_fijo)} chunks | avg {sum(len(c) for c in chunks_fijo)//len(chunks_fijo)} chars")
print(f"  Por oración (4+1):  {len(chunks_oracion)} chunks | avg {sum(len(c) for c in chunks_oracion)//len(chunks_oracion)} chars")
print(f"  Por párrafo (<600): {len(chunks_parrafo)} chunks | avg {sum(len(c) for c in chunks_parrafo)//len(chunks_parrafo)} chars")

📊 Comparación de estrategias:
  Fijo (500 chars):    15 chunks | avg 495 chars
  Por oración (4+1):  20 chunks | avg 486 chars
  Por párrafo (<600): 18 chunks | avg 410 chars


In [11]:
print("⏳ Indexando las tres estrategias...")
col_fijo    = crear_coleccion("estrategia_fija",    chunks_fijo)
col_oracion = crear_coleccion("estrategia_oracion", chunks_oracion)
col_parrafo = crear_coleccion("estrategia_parrafo", chunks_parrafo)
print("✅ Listo")

def comparar_estrategias(pregunta, top_k=1):
    print(f"\n❓ Pregunta: {pregunta}")
    print("=" * 70)
    for nombre, col in [
        ("Fijo 500 chars", col_fijo),
        ("Por oración",    col_oracion),
        ("Por párrafo",    col_parrafo)
    ]:
        resultados = recuperar(col, pregunta, top_k)
        chunk, dist = resultados[0]
        print(f"\n✂️  {nombre} (distancia: {dist:.4f}):")
        print("-" * 40)
        print(chunk[:350] + ("..." if len(chunk) > 350 else ""))

comparar_estrategias("¿Cuáles son los productos que no se pueden devolver?")
comparar_estrategias("¿Puedo pagar con Mercado Pago en cuotas sin interés?")

⏳ Indexando las tres estrategias...
✅ Listo

❓ Pregunta: ¿Cuáles son los productos que no se pueden devolver?

✂️  Fijo 500 chars (distancia: 0.7955):
----------------------------------------
dicial o resolución de organismo de defensa del consumidor.

1.2 Condiciones del producto para devolución
El producto debe presentarse en su embalaje original, sin señales de uso, con todos los accesorios incluidos y con el ticket o factura de compra. Los productos que hayan sido activados, configurados o que presenten signos evidentes de uso serán...

✂️  Por oración (distancia: 0.6311):
----------------------------------------
Las devoluciones iniciadas fuera de estos plazos no serán aceptadas bajo ninguna circunstancia, salvo que medie una orden judicial o resolución de organismo de defensa del consumidor. 1.2 Condiciones del producto para devolución
El producto debe presentarse en su embalaje original, sin señales de uso, con todos los accesorios incluidos y con el tic...

✂️  Por párrafo (dis

### 🔍 Preguntas para reflexionar

1. ¿Qué estrategia recuperó mejor la información sobre productos no devolvibles?
2. ¿El chunking por párrafo tiene ventajas para este tipo de documento (reglamento formal)?
3. ¿Qué pasaría si el documento fuera una conversación de chat en lugar de un reglamento?

> 💡 La estrategia óptima **depende del tipo de documento**. Un reglamento formal se beneficia del chunking por párrafo porque cada párrafo tiene una idea completa.

---
# 🔬 Experimento 4 — Impacto en la respuesta final del LLM

Hasta ahora comparamos solo el retrieval. Pero el chunking también afecta la **respuesta final**, porque le cambia el contexto que recibe el modelo.

In [12]:
def rag_con_chunking(pregunta, coleccion, nombre_config, top_k=2, temperature=0.2):
    resultados = recuperar(coleccion, pregunta, top_k)
    contexto = "\n\n---\n\n".join([chunk for chunk, _ in resultados])

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "Sos un asistente de atención al cliente de TechStore Argentina.\n"
                    "Respondé ÚNICAMENTE basándote en el contexto provisto.\n"
                    "Si la información no está en el contexto, decilo.\n"
                    "Respondé en español, de forma clara y concisa."
                )
            },
            {
                "role": "user",
                "content": f"CONTEXTO:\n{contexto}\n\nPREGUNTA: {pregunta}"
            }
        ],
        temperature=temperature,
        max_tokens=200
    )

    respuesta = response.choices[0].message.content
    tokens = response.usage.total_tokens

    print(f"\n⚙️  Config: {nombre_config}")
    print(f"   Tokens usados: {tokens} | Contexto: {len(contexto)} chars")
    print(f"   Respuesta: {respuesta}")
    print("-" * 60)

pregunta = "¿Qué garantía tienen las notebooks y cuándo no aplica?"
print(f"❓ Pregunta: {pregunta}")
print("=" * 60)

rag_con_chunking(pregunta, col_200,    "Chunks 200 chars")
rag_con_chunking(pregunta, col_500,    "Chunks 500 chars")
rag_con_chunking(pregunta, col_parrafo, "Chunks por párrafo")

❓ Pregunta: ¿Qué garantía tienen las notebooks y cuándo no aplica?

⚙️  Config: Chunks 200 chars
   Tokens usados: 221 | Contexto: 407 chars
   Respuesta: Las notebooks tienen una garantía de 12 meses. La garantía no aplica en casos de daños causados por mal uso, golpes, líquidos, modificaciones no autorizadas, o uso en condiciones no adecuadas.
------------------------------------------------------------

⚙️  Config: Chunks 500 chars
   Tokens usados: 359 | Contexto: 1007 chars
   Respuesta: Las notebooks tienen una garantía de 12 meses. La garantía no aplica en casos de daños causados por mal uso, golpes, líquidos, modificaciones no autorizadas, o uso en condiciones distintas a las especificadas por el fabricante.
------------------------------------------------------------

⚙️  Config: Chunks por párrafo
   Tokens usados: 313 | Contexto: 873 chars
   Respuesta: Las notebooks tienen una garantía de 12 meses. La garantía no aplica en casos de mal uso, golpes, líquidos, modificaciones 

---
# 🧪 Zona de experimentación libre

In [13]:
# ✏️ Modificá estos parámetros y observá los resultados

MI_TAMANIO  = 400
MI_OVERLAP  = 80
MI_PREGUNTA = "¿Cómo escalo mi reclamo si no me resuelven el problema?"
MI_TOP_K    = 2

mis_chunks   = chunk_fijo(CORPUS, MI_TAMANIO, overlap=MI_OVERLAP)
mi_coleccion = crear_coleccion("mi_experimento", mis_chunks)

print(f"⚙️  Config: {MI_TAMANIO} chars | overlap: {MI_OVERLAP} | top_k: {MI_TOP_K}")
print(f"   Chunks generados: {len(mis_chunks)}")

resultados = recuperar(mi_coleccion, MI_PREGUNTA, MI_TOP_K)
print(f"\n❓ Pregunta: {MI_PREGUNTA}")
print("=" * 60)
for i, (chunk, dist) in enumerate(resultados):
    print(f"\n📎 Chunk {i+1} (distancia: {dist:.4f}):")
    print(chunk)

⚙️  Config: 400 chars | overlap: 80 | top_k: 2
   Chunks generados: 24

❓ Pregunta: ¿Cómo escalo mi reclamo si no me resuelven el problema?

📎 Chunk 1 (distancia: 0.8821):
máximo de 72 horas hábiles desde su ingreso al sistema. En casos complejos que requieran investigación interna, el plazo puede extenderse hasta 10 días hábiles, notificando al cliente.

5.3 Escalamiento de reclamos
Si el cliente no está satisfecho con la resolución de su reclamo, puede solicitar que sea escalado al área de Supervisión de Atención al Cliente. Si la insatisfacción persiste, el clie

📎 Chunk 2 (distancia: 0.9790):
ea de Supervisión de Atención al Cliente. Si la insatisfacción persiste, el cliente tiene derecho a presentar su reclamo ante la Dirección de Defensa del Consumidor de su provincia o ante la plataforma nacional de resolución de conflictos. TechStore Argentina se compromete a responder todos los reclamos formales presentados ante organismos públicos dentro de los plazos legales establecidos.


In [14]:
# 🔄 Para usar tu propio corpus, reemplazá el texto abajo

MI_CORPUS = """
Pegá tu propio texto acá.
Puede ser cualquier documento que quieras explorar.
"""

if len(MI_CORPUS.strip()) < 200:
    print("⚠️  Corpus vacío o muy corto. Usando el corpus original.")
    MI_CORPUS = CORPUS

mis_chunks_custom = chunk_fijo(MI_CORPUS, 500)
mi_col_custom = crear_coleccion("corpus_custom", mis_chunks_custom)
print(f"✅ Corpus cargado: {len(MI_CORPUS)} chars → {len(mis_chunks_custom)} chunks")

mi_pregunta_custom = "¿Cuál es la idea principal del documento?"
resultados = recuperar(mi_col_custom, mi_pregunta_custom, top_k=2)

print(f"\n❓ Pregunta: {mi_pregunta_custom}")
for i, (chunk, dist) in enumerate(resultados):
    print(f"\n📎 Chunk {i+1} (dist: {dist:.4f}): {chunk[:300]}")

⚠️  Corpus vacío o muy corto. Usando el corpus original.
✅ Corpus cargado: 7434 chars → 15 chunks

❓ Pregunta: ¿Cuál es la idea principal del documento?

📎 Chunk 1 (dist: 1.5580): REGLAMENTO INTERNO DE OPERACIONES — TECHSTORE ARGENTINA S.A.
Versión 3.2 — Actualizado enero 2025

SECCIÓN 1: POLÍTICA DE DEVOLUCIONES Y CAMBIOS

1.1 Plazo de devolución
Los clientes tienen derecho a solicitar la devolución de productos dentro de los 30 días corridos contados desde la fecha de entre

📎 Chunk 2 (dist: 1.6106): dicial o resolución de organismo de defensa del consumidor.

1.2 Condiciones del producto para devolución
El producto debe presentarse en su embalaje original, sin señales de uso, con todos los accesorios incluidos y con el ticket o factura de compra. Los productos que hayan sido activados, configur


---
# ✅ Resumen del Colab 1

| Variable | Lo que aprendimos |
|---|---|
| **Tamaño chico (200)** | Alta precisión, puede faltar contexto |
| **Tamaño medio (500)** | Buen balance para documentos formales |
| **Tamaño grande (1000)** | Más contexto pero mezcla temas |
| **Sin overlap** | Riesgo de partir información en el borde |
| **Con overlap** | Mejor cobertura, más chunks y más costo |
| **Fijo por chars** | Simple pero corta oraciones a la mitad |
| **Por oración** | Respeta unidades de significado |
| **Por párrafo** | Ideal para documentos formales estructurados |

## 🚀 Siguiente: Colab 2 — Embeddings
¿Qué diferencia hay entre buscar por TF-IDF (que ya conocen) y buscar por embeddings semánticos?